# Linear Regression — Built from Scratch

This notebook implements **Ordinary Least Squares (OLS) Linear Regression** entirely from scratch using only NumPy,
then evaluates it on the pre-processed student performance dataset.

OLS minimises the residual sum of squares:

$$\hat{\beta} = \arg\min_\beta \; \|y - X\beta\|_2^2$$

which has the closed-form solution:

$$\hat{\beta} = (X^TX)^{-1} X^T y$$

| Task | Target | Metric |
|------|--------|--------|
| Regression | `exam_score` | RMSE, MAE, R² |

Mirrors the structure of `ridge_regression.ipynb` for easy side-by-side comparison.

---

## 1 · Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings

from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROCESSED_DIR = './processed'
OUT_DIR       = '../project/outputs/linear_regression'
os.makedirs(f'{OUT_DIR}/figures', exist_ok=True)
os.makedirs(f'{OUT_DIR}/models',  exist_ok=True)

print('Config OK. Output directory:', OUT_DIR)

---
## 2 · Load Pre-Processed Artefacts

In [ ]:
# Scaled splits — linear models require standardised features
X_train = pd.read_csv(f'{PROCESSED_DIR}/X_train_scaled.csv', index_col=0)
X_val   = pd.read_csv(f'{PROCESSED_DIR}/X_val_scaled.csv',   index_col=0)
X_test  = pd.read_csv(f'{PROCESSED_DIR}/X_test_scaled.csv',  index_col=0)

# Regression targets
y_reg_train = pd.read_csv(f'{PROCESSED_DIR}/y_reg_train.csv', index_col=0).squeeze()
y_reg_val   = pd.read_csv(f'{PROCESSED_DIR}/y_reg_val.csv',   index_col=0).squeeze()
y_reg_test  = pd.read_csv(f'{PROCESSED_DIR}/y_reg_test.csv',  index_col=0).squeeze()

print('Loaded splits:')
for name, arr in [('Train', X_train), ('Val', X_val), ('Test', X_test)]:
    print(f'  {name:5s}  {arr.shape[0]:6d} rows  {arr.shape[1]} features')
print(f'\nRegression target  — mean={y_reg_train.mean():.2f}, std={y_reg_train.std():.2f}')

In [ ]:
# Encode any remaining string columns
from sklearn.preprocessing import OrdinalEncoder

cat_cols = X_train.select_dtypes(include='object').columns.tolist()

if cat_cols:
    print(f"Encoding categorical columns: {cat_cols}")
    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_train[cat_cols] = enc.fit_transform(X_train[cat_cols])
    X_val[cat_cols]   = enc.transform(X_val[cat_cols])
    X_test[cat_cols]  = enc.transform(X_test[cat_cols])
else:
    print("No categorical columns found — all good.")

In [ ]:
# Drop columns that are entirely NaN
cols_to_drop = ['stress_level', 'parental_support_level', 'motivation_level', 'stress_anxiety_composite']
cols_to_drop = [c for c in cols_to_drop if c in X_train.columns]  # only drop if present

X_train = X_train.drop(columns=cols_to_drop)
X_val   = X_val.drop(columns=cols_to_drop)
X_test  = X_test.drop(columns=cols_to_drop)

feature_names = X_train.columns.tolist()

print(f"Dropped columns : {cols_to_drop}")
print(f"Remaining features: {X_train.shape[1]}")
print(f"Any NaNs left: {X_train.isna().sum().sum()}")

---
## 3 · Linear Regression — Built from Scratch

### 3.1 · The Maths

**Ordinary Least Squares (OLS)** minimises the residual sum of squares:
$$\mathcal{L}_{OLS} = \|y - X\beta\|_2^2$$

Setting the gradient to zero and solving analytically:
$$\nabla_\beta \mathcal{L} = -2X^T(y - X\beta) = 0$$
$$\Rightarrow \hat{\beta} = (X^TX)^{-1} X^T y$$

Key properties:
- No regularisation — coefficients are unconstrained
- Produces the minimum-variance unbiased estimator (Gauss-Markov theorem)
- Can overfit when features are correlated or when there are many features relative to rows
- Compare against `ridge_regression.ipynb` to see the effect of adding L2 regularisation

In [ ]:
class LinearRegressionScratch:
    """
    Ordinary Least Squares Linear Regression using the closed-form solution:

        beta = (X'X)^{-1} X'y

    An intercept column (bias) is prepended to X internally.
    No regularisation is applied.
    """

    def __init__(self):
        self.beta_ = None   # shape: (n_features + 1,)  [intercept first]

    # ── Internal helpers ────────────────────────────────────────────────────
    @staticmethod
    def _add_bias(X):
        """Prepend a column of ones to X."""
        ones = np.ones((X.shape[0], 1))
        return np.hstack([ones, X])

    # ── Fit ─────────────────────────────────────────────────────────────────
    def fit(self, X, y):
        """
        Compute the closed-form OLS solution.

        Uses np.linalg.solve rather than explicit matrix inversion
        for numerical stability.
        """
        X_b = self._add_bias(np.asarray(X, dtype=float))
        y_  = np.asarray(y, dtype=float)

        # beta = (X'X)^{-1} X'y
        A = X_b.T @ X_b
        b = X_b.T @ y_
        self.beta_ = np.linalg.solve(A, b)   # more stable than explicit inv
        return self

    # ── Predict ─────────────────────────────────────────────────────────────
    def predict(self, X):
        X_b = self._add_bias(np.asarray(X, dtype=float))
        return X_b @ self.beta_

    # ── Convenience properties ───────────────────────────────────────────────
    @property
    def intercept_(self):
        return self.beta_[0]

    @property
    def coef_(self):
        return self.beta_[1:]


# Quick sanity check against sklearn
from sklearn.linear_model import LinearRegression as SklearnLR

scratch = LinearRegressionScratch().fit(X_train, y_reg_train)
sk      = SklearnLR().fit(X_train, y_reg_train)

scratch_rmse = np.sqrt(mean_squared_error(y_reg_val, scratch.predict(X_val)))
sk_rmse      = np.sqrt(mean_squared_error(y_reg_val, sk.predict(X_val)))

print(f'Scratch  val RMSE: {scratch_rmse:.6f}')
print(f'Sklearn  val RMSE: {sk_rmse:.6f}')
print(f'Max coeff difference: {np.abs(scratch.coef_ - sk.coef_).max():.2e}')   # should be ~1e-8

---
## 4 · Regression Task — Predict `exam_score`

### 4.1 · Fit on Train + Val, Evaluate on Test

OLS has no hyperparameter to tune, so we go straight to fitting on the
combined train + val set and evaluating on the held-out test set.

In [ ]:
# Combine train + val for final fit (same convention as ridge_regression.ipynb)
X_trainval     = pd.concat([X_train, X_val])
y_reg_trainval = pd.concat([y_reg_train, y_reg_val])

best_ols = LinearRegressionScratch()
best_ols.fit(X_trainval, y_reg_trainval)

y_reg_pred = best_ols.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))
mae  = mean_absolute_error(y_reg_test, y_reg_pred)
r2   = r2_score(y_reg_test, y_reg_pred)

print('─' * 35)
print(f'  RMSE     : {rmse:.3f}')
print(f'  MAE      : {mae:.3f}')
print(f'  R²       : {r2:.4f}')
print('─' * 35)

### 4.2 · Actual vs Predicted Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter
axes[0].scatter(y_reg_test, y_reg_pred, alpha=0.3, s=8, color='steelblue')
lims = [y_reg_test.min(), y_reg_test.max()]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual exam_score')
axes[0].set_ylabel('Predicted exam_score')
axes[0].set_title(f'Actual vs Predicted  (R²={r2:.3f})')
axes[0].legend()

# Residuals
residuals = y_reg_test.values - y_reg_pred
axes[1].scatter(y_reg_pred, residuals, alpha=0.3, s=8, color='coral')
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xlabel('Predicted exam_score')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residual Plot')

plt.suptitle('OLS Linear Regression — Test Set Performance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/reg_actual_vs_predicted.png', dpi=150)
plt.show()

### 4.3 · Feature Importances (Coefficient Magnitudes)

Because features are standardised, coefficient magnitude is a direct proxy for importance.

In [ ]:
importances = pd.Series(np.abs(best_ols.coef_), index=feature_names)
top20 = importances.sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 6))
top20.plot(kind='barh', color='steelblue')
plt.gca().invert_yaxis()
plt.xlabel('|Coefficient| (standardised features)')
plt.title('OLS Linear Regression — Top 20 Feature Importances')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/reg_feature_importances.png', dpi=150)
plt.show()

### 4.4 · Residual Distribution

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(residuals, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
plt.axvline(0, color='red', linestyle='--', lw=1.5)
plt.xlabel('Residual (Actual − Predicted)')
plt.ylabel('Count')
plt.title('OLS Linear Regression — Residual Distribution')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/reg_residual_distribution.png', dpi=150)
plt.show()
print(f'Residual mean : {residuals.mean():.4f}')
print(f'Residual std  : {residuals.std():.4f}')

### 4.5 · Save Model

In [ ]:
import joblib
joblib.dump(best_ols, f'{OUT_DIR}/models/linear_reg.joblib')
print(f'OLS model saved to {OUT_DIR}/models/linear_reg.joblib')

---
## 5 · Comparison with Ridge Regression

Run `ridge_regression.ipynb` first to populate `ridge_rmse`, `ridge_mae`, and `ridge_r2`.
Or enter the Ridge test-set metrics manually below.

In [ ]:
# Fill in Ridge results from ridge_regression.ipynb (or leave as None to skip)
ridge_rmse = 10.673
ridge_mae  = 8.773
ridge_r2   = 0.1772
ridge_lam  = 142.0831

summary = pd.DataFrame([
    {
        'Model'    : 'OLS (no regularisation)',
        'λ'        : '—',
        'Test RMSE': f'{rmse:.3f}',
        'Test MAE' : f'{mae:.3f}',
        'Test R²'  : f'{r2:.4f}',
    },
    {
        'Model'    : 'Ridge',
        'λ'        : f'{ridge_lam:.4f}',
        'Test RMSE': f'{ridge_rmse:.3f}',
        'Test MAE' : f'{ridge_mae:.3f}',
        'Test R²'  : f'{ridge_r2:.4f}',
    },
])
display(summary)

print('\nAll figures saved to:', f'{OUT_DIR}/figures/')
print('All models saved to: ', f'{OUT_DIR}/models/')